# Notebook 05 – Tín hiệu cảnh báo sớm: hành vi trước khi vỡ nợ

Ngoài xác suất vỡ nợ, notebook này tìm các **hành vi thường xuất hiện trước nợ xấu** có thể đo được từ dữ liệu hiện có:

| Nhóm hành vi | Ý nghĩa nghiệp vụ |
|---|---|
| **Trễ hạn thanh toán** | số ngày quá hạn (DPD), tỷ lệ kỳ trả muộn, số kỳ có DPD trong lịch sử |
| **Vay nhiều nơi** (loan stacking) | nhiều hợp đồng đang hoạt động, nhiều tổ chức cho vay, nhiều lần tra cứu/nộp hồ sơ gần đây |
| **Gánh nặng nợ** | dư nợ, khoản trả hàng kỳ, tỷ lệ trên thu nhập |
| **Vay mới để trả nợ cũ** (đảo nợ) | mở hợp đồng mới ngay trước khi nộp hồ sơ, hồ sơ vay dồn dập, các trường tái cơ cấu / gia hạn nếu có |

**Lưu ý phương pháp.** Dữ liệu chỉ có một nhãn là vỡ nợ (`target`); không có nhãn "sẽ trễ hạn" hay "sẽ đảo nợ" trong tương lai. Vì vậy các hành vi trên được dùng như **chỉ báo dẫn trước** đo tại thời điểm nộp hồ sơ (chỉ dùng thông tin trước ngày quyết định), và được đánh giá bằng mức độ chúng báo trước vỡ nợ trên tập test.

Nhóm biến được chọn **tự động theo mô tả cột** trong `feature_definitions.csv` (không đoán theo tên cột) và in ra để kiểm tra lại.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings, re; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt
import utils
from utils import META
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
SEED = 42
pd.set_option("display.max_colwidth", 90)
defs = utils.load_definitions()
print("So cot co mo ta:", len(defs))

## 1. Định nghĩa nhóm hành vi theo mô tả cột

In [ ]:
GROUPS = utils.BEHAVIOR_GROUPS               # regex theo mo ta cot, dinh nghia trong utils.py
assign_group = utils.assign_group
GROUP_VN = {"tre_han": "Tre han", "vay_nhieu_noi": "Vay nhieu noi", "ganh_nang_no": "Ganh nang no", "dao_no": "Dao no / tai co cau"}
for g, p in GROUPS.items(): print(f"{GROUP_VN[g]:<22} {p}")

df = pl.read_parquet(OUT_DIR / "features_v1.parquet")
feats = [c for c in df.columns if c not in META]
desc = utils.describe(feats, defs)
grp = desc.map(assign_group)
print(grp.value_counts(dropna=False).rename("so_bien"))

In [ ]:
iv = pd.read_csv(OUT_DIR / "iv_table.csv", index_col=0)["IV"]
tab = pd.DataFrame({"nhom": grp, "IV": iv.reindex(feats), "mo_ta": desc}).dropna(subset=["nhom"])
for g in GROUPS:
    t = tab[tab["nhom"] == g].sort_values("IV", ascending=False)
    print(f"\n=== {GROUP_VN[g]}: {len(t)} bien | 8 bien IV cao nhat"); display(t.head(8).round(4))

## 2. Bổ sung đặc trưng hành vi từ lịch sử tín dụng bên ngoài (`credit_bureau_a_1`)
Bảng này (≈ 16 triệu dòng, nhiều hợp đồng/hồ sơ) chưa được dùng ở notebook 02 vì nặng. Ở đây chỉ lấy các cột liên quan đến bốn nhóm hành vi và gộp thành:
- số hợp đồng, **số tổ chức cho vay khác nhau** (vay nhiều nơi),
- DPD / số tiền quá hạn lớn nhất, trung bình (trễ hạn),
- tổng dư nợ, tổng khoản trả hàng kỳ (gánh nặng nợ),
- **số hợp đồng mở trong 30/90/180 ngày trước ngày nộp hồ sơ** (dấu hiệu vay mới dồn dập / vay để trả nợ cũ).

In [ ]:
base = utils.load_base()
cbf, cb_info = utils.credit_bureau_features(base, defs)          # ham dung chung voi notebook 07 (tap test)
print("Cot so:", len(cb_info["num"]), "| cot phan loai:", cb_info["cat"], "| cot ngay mo hop dong:", cb_info["open"])
display(pd.Series({c: defs.get(c, "") for c in cb_info["num"] + cb_info["cat"] + cb_info["open"]}, name="mo_ta").to_frame())
cbf = utils.prune_features(cbf, max_null=0.95)
print("Dac trung moi tu credit_bureau_a_1:", cbf.width - 1)

# Cung quy tac troi do san co nhu notebook 02 (train vs valid, khong dung test)
weeks = np.sort(df["WEEK_NUM"].unique()); n_w = len(weeks)
wk = df.select("case_id", "WEEK_NUM")
cbw = cbf.join(wk, on="case_id", how="right")
w_tr, w_va = weeks[: int(n_w * 0.70)].tolist(), weeks[int(n_w * 0.70): int(n_w * 0.85)].tolist()
cols = [c for c in cbf.columns if c != "case_id"]
n_tr = cbw.filter(pl.col("WEEK_NUM").is_in(w_tr)).select([pl.col(c).is_null().mean().alias(c) for c in cols]).row(0)
n_va = cbw.filter(pl.col("WEEK_NUM").is_in(w_va)).select([pl.col(c).is_null().mean().alias(c) for c in cols]).row(0)
drop_cb = [c for c, a, b in zip(cols, n_tr, n_va) if abs(b - a) >= 0.30]
cbf = cbf.drop(drop_cb)
print("Loai do troi do san co:", len(drop_cb), "| con lai:", cbf.width - 1)

In [ ]:
# Gan nhom hanh vi cho dac trung moi
new_feats = [c for c in cbf.columns if c != "case_id"]
grp_new = pd.Series({c: utils.cb_group(c, defs) for c in new_feats})
grp_all = pd.concat([grp, grp_new])

dfx = df.join(cbf, on="case_id", how="left")
pdf = dfx.to_pandas()
train, valid, test = utils.time_split(pdf)
feats_all = feats + new_feats
print("Tong dac trung:", len(feats_all), "| train/valid/test:", len(train), len(valid), len(test))
print(grp_new.value_counts(dropna=False).rename("so_bien_moi"))

## 3. Chỉ số cảnh báo sớm theo từng nhóm hành vi
Mỗi nhóm được tóm thành một **điểm hành vi**: Logistic Regression trên biến WoE của riêng nhóm đó (fit trên train).
Cờ cảnh báo của nhóm bật khi điểm thuộc **20% rủi ro nhất** (ngưỡng cố định từ train). Đánh giá trên test:
độ phủ (% hồ sơ bị gắn cờ), tỷ lệ vỡ nợ khi có / không có cờ, **lift** (= tỷ lệ vỡ nợ khi có cờ ÷ tỷ lệ chung) và AUC của riêng điểm nhóm.

In [ ]:
tr_s = train.sample(min(len(train), 300_000), random_state=SEED)
base_rate = test["target"].mean()
scores, rows = {}, []
for g in GROUPS:
    cols = [c for c in feats_all if grp_all.get(c) == g]
    if not cols:
        print("Bo qua nhom", g, "- khong co bien"); continue
    m = Pipeline([("woe", utils.WoEEncoder()), ("sc", StandardScaler()),
                  ("lr", LogisticRegression(max_iter=1000, C=0.1))]).fit(tr_s[cols], tr_s["target"])
    s_tr = m.predict_proba(tr_s[cols])[:, 1]; thr = np.quantile(s_tr, 0.80)
    s_te = m.predict_proba(test[cols])[:, 1]
    # nhieu ho so cung diem (bien thua, phan lon rong) -> dung '>' de co khong bat cho ca nhom diem trung
    flag = (s_te >= thr) if (s_tr >= thr).mean() <= 0.30 else (s_te > thr)
    scores[g] = s_te; test["co_" + g] = flag.astype(int)
    rows.append({"nhom": GROUP_VN[g], "so_bien": len(cols), "AUC_test": roc_auc_score(test["target"], s_te),
                 "do_phu": flag.mean(), "vo_no_khi_co_co": test.loc[flag, "target"].mean(),
                 "vo_no_khi_khong": test.loc[~flag, "target"].mean(),
                 "lift": test.loc[flag, "target"].mean() / base_rate})
ewi = pd.DataFrame(rows).set_index("nhom")
ewi.to_csv(OUT_DIR / "early_warning_groups.csv")
print("Ty le vo no chung (test): %.2f%%" % (100 * base_rate)); ewi.round(4)

## 4. Hồ sơ hành vi: số cờ cảnh báo và tổ hợp hành vi
Nếu các hành vi thực sự là tín hiệu dẫn trước, tỷ lệ vỡ nợ phải **tăng dần theo số cờ** đồng thời bật.

In [ ]:
flag_cols = [c for c in test.columns if c.startswith("co_")]
test["so_co"] = test[flag_cols].sum(axis=1)
prof = test.groupby("so_co").agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"))
prof["ty_trong"] = prof["so_ho_so"] / len(test); prof["lift"] = prof["ty_le_vo_no"] / base_rate
display(prof.round(4))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(prof.index.astype(str), 100 * prof["ty_le_vo_no"]); ax.axhline(100 * base_rate, ls="--", c="gray")
ax.set_xlabel("So co canh bao dong thoi"); ax.set_ylabel("% vo no (test)"); ax.set_title("Ty le vo no theo so hanh vi rui ro"); plt.show()

combo = test[flag_cols].astype(str).agg("".join, axis=1).map(
    lambda s: " + ".join(GROUP_VN[c[3:]] for c, v in zip(flag_cols, s) if v == "1") or "(khong co co)")
cmb = test.assign(to_hop=combo).groupby("to_hop").agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"))
cmb = cmb[cmb["so_ho_so"] >= 300].sort_values("ty_le_vo_no", ascending=False)
cmb["lift"] = cmb["ty_le_vo_no"] / base_rate
cmb.to_csv(OUT_DIR / "early_warning_combos.csv"); cmb.head(12).round(4)

## 5. Đặc trưng hành vi bổ sung có cải thiện mô hình vỡ nợ không?
So sánh LightGBM trên bộ đặc trưng gốc và bộ có thêm đặc trưng `credit_bureau_a_1`, cùng tham số và cùng cách chia tập.

In [ ]:
P = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")
res, models = [], {}
for name, cols in [("Dac trung goc", feats), ("Goc + hanh vi credit bureau", feats_all)]:
    m = lgb.LGBMClassifier(**P).fit(train[cols], train["target"], eval_set=[(valid[cols], valid["target"])],
                                     callbacks=[lgb.early_stopping(200, verbose=False)])
    p_te = m.predict_proba(test[cols])[:, 1]
    gw = utils.gini_by_week(test.assign(score=p_te))
    res.append({"bo_dac_trung": name, "so_bien": len(cols), "best_iteration": m.best_iteration_,
                "AUC_valid": roc_auc_score(valid["target"], m.predict_proba(valid[cols])[:, 1]),
                "AUC_test": roc_auc_score(test["target"], p_te), "Gini_tuan_tb": gw.mean(), "Gini_tuan_min": gw.min()})
    models[name] = (m, cols)
res = pd.DataFrame(res).set_index("bo_dac_trung"); res.to_csv(OUT_DIR / "behavior_feature_gain.csv"); display(res.round(4))

# Kiem dinh DeLong: chenh lech AUC test giua hai bo dac trung co y nghia thong ke khong
p_base = models["Dac trung goc"][0].predict_proba(test[feats])[:, 1]
p_full = models["Goc + hanh vi credit bureau"][0].predict_proba(test[feats_all])[:, 1]
dl = utils.delong_test(test["target"], p_full, p_base)
print("DeLong (goc + credit bureau vs goc): chenh AUC = %.4f | z = %.2f | p = %.2e" % (dl["chenh"], dl["z"], dl["p_value"]))

# Luu bo dac trung mo rong, mo hinh cuoi va du doan cho notebook 06-07
import joblib
FINAL = "LGBM | goc + credit bureau"
fm, fcols = models["Goc + hanh vi credit bureau"]
joblib.dump({"model": fm, "features": fcols, "name": FINAL}, OUT_DIR / "final_model.joblib")
for split, d in [("valid", valid), ("test", test)]:
    p = fm.predict_proba(d[fcols])[:, 1]
    d[["case_id", "WEEK_NUM", "target"]].assign(pd_raw=p).to_parquet(OUT_DIR / f"preds_final_{split}.parquet", index=False)
    allp = pd.read_parquet(OUT_DIR / f"preds_all_{split}.parquet")
    allp = allp.merge(d[["case_id"]].assign(**{FINAL: p}), on="case_id", how="left")
    allp.to_parquet(OUT_DIR / f"preds_all_{split}.parquet", index=False)
dfx.write_parquet(OUT_DIR / "features_v2.parquet")
pd.Series(new_feats, name="cot").to_csv(OUT_DIR / "features_v2_new_cols.csv", index=False)
grp_all.rename("nhom").to_csv(OUT_DIR / "behavior_groups.csv")
print("Da luu features_v2.parquet", dfx.shape)

In [ ]:
# Dong gop (gain) cua tung nhom hanh vi trong mo hinh day du
m, cols = models["Goc + hanh vi credit bureau"]
gain = pd.Series(m.booster_.feature_importance("gain"), index=cols)
share = (gain.groupby(grp_all.reindex(cols).fillna("khac").map(lambda g: GROUP_VN.get(g, "Khac"))).sum() / gain.sum()).sort_values()
share.plot.barh(figsize=(6, 3), title="Ty trong gain theo nhom hanh vi"); plt.show()
print((100 * share).round(1).astype(str) + "%")
print("\nTop 15 dac trung:"); display(pd.DataFrame({"gain_%": 100 * gain / gain.sum(), "nhom": grp_all.reindex(cols),
                                                     "mo_ta": utils.describe(cols, defs)}).sort_values("gain_%", ascending=False).head(15).round(2))

## 6. Nhận xét (tự viết)
- Nhóm hành vi báo trước vỡ nợ mạnh nhất (lift, AUC): ...
- Tỷ lệ vỡ nợ theo số cờ cảnh báo – có tăng đơn điệu không: ...
- Tổ hợp hành vi nguy hiểm nhất (ví dụ trễ hạn + vay nhiều nơi): ...
- Đặc trưng từ lịch sử tín dụng bên ngoài có cải thiện AUC / độ ổn định không: ...
- Ứng dụng: dùng cờ cảnh báo để **giám sát sau giải ngân** (theo dõi lại hồ sơ khi cờ bật), giảm hạn mức hoặc yêu cầu xác minh thêm.
- Hạn chế: không có nhãn trễ hạn/đảo nợ trong tương lai; nhóm biến chọn theo từ khoá mô tả nên cần rà soát bảng ở mục 1.